# Vulnerability data → model parameters

Queries the three authoritative public sources and turns them into parameters for the ODE model in
`ODEtoVuln_daily.ipynb`:

| Source | What it provides here |
|---|---|
| **NVD** (`services.nvd.nist.gov/rest/json/cves/2.0`) | the CVE corpus, publication rates, CVSS attack vectors, and the CISA KEV flags/dates mirrored into each record |
| **CISA KEV** (`known_exploited_vulnerabilities.json`) | which CVEs are known-exploited, when CISA added them, and the federal remediation deadline |
| **MITRE** (`cveawg.mitre.org/api/cve/…`) | per-CVE CNA record: reservation and publication timestamps, used to sanity-check disclosure timing |

Two populations are measured: **all software** (the global ecosystem) and **one CPE family**, Microsoft
SharePoint, as a stand-in for enterprise server software.

Every response is cached under `data/`, so re-running is free and the derived parameters are reproducible.
NVD's anonymous rate limit is 5 requests / 30 s; the fetcher self-throttles to 6.6 s between calls. Get an
API key from nvd.nist.gov to go faster.

**Snapshot 2026-08-14.** This notebook writes three files the model notebook reads: `data/derived_params.json` (parameters), `data/global_history.json` (per-year publication rates, for the transient runs) and `data/sharepoint_union.json` (the de-duplicated CVE set for the CPE family).

In [1]:
import datetime as dt
import json
import statistics as st
import time
import urllib.parse
import urllib.request
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)
NVD = "https://services.nvd.nist.gov/rest/json/cves/2.0"
CPE_API = "https://services.nvd.nist.gov/rest/json/cpes/2.0"
KEV_FEED = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
MITRE = "https://cveawg.mitre.org/api/cve/"
UA = {"User-Agent": "vulnapocalypse-research/1.0"}

AS_OF = dt.date(2026, 8, 14)     # the snapshot these numbers describe
TAG = AS_OF.isoformat()          # cache keys carry it, so snapshots never overwrite each other
NVD_GAP = 6.6                    # seconds between calls: 5 requests / 30 s anonymous limit
_last_call = [0.0]


def fetch(url, cache, retries=4):
    """GET with on-disk caching and rate limiting; cached responses cost nothing."""
    path = DATA / cache
    if path.exists():
        return json.loads(path.read_text())
    for attempt in range(retries):
        wait = NVD_GAP - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=180) as r:
                payload = json.loads(r.read())
            _last_call[0] = time.time()
            path.write_text(json.dumps(payload))
            return payload
        except Exception as exc:                                    # noqa: BLE001
            print(f"  retry {attempt + 1}/{retries} on {cache}: {type(exc).__name__} "
                  f"{getattr(exc, 'code', '')}")
            _last_call[0] = time.time()
            time.sleep(15)
    raise RuntimeError(f"could not fetch {cache}")


def nvd(cache, flags=(), **params):
    """NVD query; `flags` are valueless parameters such as hasKev."""
    url = NVD + "?" + urllib.parse.urlencode(params, safe=":")
    for flag in flags:
        url += f"&{flag}"
    return fetch(url, cache)


day = lambda stamp: dt.date.fromisoformat(stamp[:10])
print("helpers ready · cache:", DATA.resolve())

helpers ready · cache: /home/jo5iah/vulnapocalypse/data


## 1. The global corpus and its current publication rate

NVD caps any single publication-date range at 120 days, so a year of publications takes four windows. The
publication rate is the model's $\gamma + \sigma S$ — the two are additive in the equations and NVD cannot
distinguish "found in old code" from "shipped broken last month," so the measurement is their sum.

In [2]:
corpus_total = nvd(f"nvd_total_{TAG}.json", resultsPerPage=1)["totalResults"]

windows = []
for i in range(4):
    end = AS_OF - dt.timedelta(days=91 * i)
    start = end - dt.timedelta(days=91)
    n = nvd(f"nvd_window_{TAG}_{i}.json", resultsPerPage=1,
            pubStartDate=f"{start}T00:00:00.000", pubEndDate=f"{end}T23:59:59.999")["totalResults"]
    windows.append({"start": str(start), "end": str(end), "count": n})
    print(f"  {start} .. {end}   {n:>6,} CVEs   {n / 91:6.1f}/day")

year_count = sum(w["count"] for w in windows)
gamma_global = year_count / 364.0
gamma_global_recent = windows[0]["count"] / 91.0

print(f"\nNVD corpus: {corpus_total:,} CVEs")
print(f"last 364 days: {year_count:,} → {gamma_global:.0f}/day")
print(f"most recent 91 days: {windows[0]['count']:,} → {gamma_global_recent:.0f}/day "
      f"({gamma_global_recent / gamma_global:.1f}× the trailing year)")

  2026-05-15 .. 2026-08-14   26,182 CVEs    287.7/day
  2026-02-13 .. 2026-05-15   18,657 CVEs    205.0/day
  2025-11-14 .. 2026-02-13   14,497 CVEs    159.3/day
  2025-08-15 .. 2025-11-14   12,683 CVEs    139.4/day

NVD corpus: 377,221 CVEs
last 364 days: 72,019 → 198/day
most recent 91 days: 26,182 → 288/day (1.5× the trailing year)


## 2. Exploitation: the CISA KEV catalog

KEV is the only public, curated record of *observed* exploitation. Two properties matter for the model.

It is **cumulative** — entries are essentially never withdrawn — so the catalog size is total exploitation
influx over KEV's lifetime, *not* the model's $E$ (which has an outflow). What KEV gives directly is the
**influx rate** into $E$ and the **lag** between disclosure and observed exploitation.

It is also a **lower bound**: KEV requires reliable evidence of exploitation and relevance to federal
networks, so quiet or narrowly-targeted exploitation is missing. Every exploitation parameter derived
here inherits that downward bias.

In [3]:
kev_catalog = fetch(KEV_FEED, f"cisa_kev_{TAG}.json")
kev_nvd = nvd(f"nvd_kev_{TAG}.json", flags=("hasKev",), resultsPerPage=2000)["vulnerabilities"]

kev_by_year, lags, due_windows = {}, [], []
for v in kev_nvd:
    c = v["cve"]
    added, published = day(c["cisaExploitAdd"]), day(c["published"])
    kev_by_year[added.year] = kev_by_year.get(added.year, 0) + 1
    lags.append((added - published).days)
    if c.get("cisaActionDue"):
        due_windows.append((day(c["cisaActionDue"]) - added).days)

full_years = [y for y in sorted(kev_by_year) if 2022 <= y <= 2025]
kev_rate_global = st.mean(kev_by_year[y] for y in full_years) / 365.0
zero_day_share = sum(1 for l in lags if l <= 0) / len(lags)

print(f"KEV catalog: {kev_catalog['count']:,} entries (released {kev_catalog['dateReleased'][:10]})")
print(f"  = {len(kev_nvd) / corpus_total:.3%} of the NVD corpus")
print(f"additions per year: " + ", ".join(f"{y}:{kev_by_year[y]}" for y in sorted(kev_by_year)))
print(f"  {full_years[0]}–{full_years[-1]} mean: {kev_rate_global * 365:.0f}/yr = "
      f"{kev_rate_global:.3f}/day  ← exploitation influx")
print(f"disclosure → KEV lag: median {st.median(lags):.0f} d, p10 {sorted(lags)[len(lags) // 10]} d, "
      f"p90 {sorted(lags)[9 * len(lags) // 10]:,} d")
print(f"  exploited on or before publication (zero-day path): {zero_day_share:.1%}")
print(f"CISA remediation deadline (due − added): median {st.median(due_windows):.0f} days")

KEV catalog: 1,665 entries (released 2026-08-11)
  = 0.441% of the NVD corpus
additions per year: 2021:311, 2022:555, 2023:187, 2024:186, 2025:245, 2026:181
  2022–2025 mean: 293/yr = 0.803/day  ← exploitation influx
disclosure → KEV lag: median 271 d, p10 0 d, p90 2,660 d
  exploited on or before publication (zero-day path): 12.9%
CISA remediation deadline (due − added): median 21 days


## 3. Reachability, from CVSS attack vector

The model's $\varepsilon$ is the fraction of vulnerabilities that are actually reachable by an attacker.
The closest public proxy is the CVSS attack vector: NETWORK or ADJACENT_NETWORK means remotely reachable
given a deployed, exposed instance. This *overstates* $\varepsilon$ for a hardened estate and understates
it for an internet-facing one, so treat it as an upper anchor for a given population.

In [4]:
def network_share(vulns):
    """Fraction of CVEs whose CVSS attack vector is network-reachable, and the sample size."""
    scored = reachable = 0
    for v in vulns:
        metrics = v["cve"].get("metrics", {})
        for key in ("cvssMetricV40", "cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
            if key in metrics:
                data = metrics[key][0]["cvssData"]
                vector = (data.get("attackVector") or data.get("accessVector") or "").upper()
                scored += 1
                reachable += vector in ("NETWORK", "ADJACENT_NETWORK")
                break
    return (reachable / scored if scored else float("nan")), scored


eps_kev, n_kev = network_share(kev_nvd)
print(f"KEV population: {eps_kev:.1%} network-reachable (n={n_kev:,})")

KEV population: 75.0% network-reachable (n=1,665)


## 4. One CPE family: Microsoft SharePoint

SharePoint has no single CPE — NVD spreads it across `sharepoint_server`, `sharepoint_foundation`,
`sharepoint_enterprise_server` and older names, and a CVE typically lists several. The CPE dictionary is
queried first to discover the product names, then each is swept and the results de-duplicated by CVE ID.

In [5]:
cpe_hits = fetch(CPE_API + "?keywordSearch=sharepoint&resultsPerPage=2000", "cpe_sharepoint.json")
products = sorted({c["cpe"]["cpeName"].split(":")[4] for c in cpe_hits["products"]
                   if c["cpe"]["cpeName"].split(":")[3] == "microsoft"})
print("SharePoint product names in the CPE dictionary:")
print("  " + ", ".join(products))

TARGETS = ["sharepoint_server", "sharepoint_foundation", "sharepoint_enterprise_server",
           "sharepoint_services", "sharepoint_portal_server", "sharepoint_team_services",
           "office_sharepoint_server", "sharepoint_designer"]

sharepoint = {}
for product in TARGETS:
    hits = nvd(f"nvd_sp_{product}_{TAG}.json", resultsPerPage=2000,
               virtualMatchString=f"cpe:2.3:a:microsoft:{product}")
    for v in hits["vulnerabilities"]:
        sharepoint[v["cve"]["id"]] = v
    print(f"  {product:<30} {hits['totalResults']:>4} CVEs   union {len(sharepoint):>4}")

# the de-duplicated union is an output, not scratch: the model notebook reads it
Path("data/sharepoint_union.json").write_text(json.dumps({"as_of": str(AS_OF),
                                                         "vulnerabilities": list(sharepoint.values())}))
sp = list(sharepoint.values())
pubs = sorted(day(v["cve"]["published"]) for v in sp)
sp_365 = sum(1 for d in pubs if (AS_OF - d).days <= 365)
sp_1095 = sum(1 for d in pubs if (AS_OF - d).days <= 1095)
gamma_sp, gamma_sp_recent = sp_1095 / 1095.0, sp_365 / 365.0

sp_kev = [v for v in sp if v["cve"].get("cisaExploitAdd")]
sp_lags = [(day(v["cve"]["cisaExploitAdd"]) - day(v["cve"]["published"])).days for v in sp_kev]
kev_rate_sp = len(sp_kev) / ((AS_OF - day(min(v["cve"]["cisaExploitAdd"] for v in sp_kev))).days)
eps_sp, n_sp = network_share(sp)

per_year = {}
for d in pubs:
    per_year[d.year] = per_year.get(d.year, 0) + 1

print(f"\n{len(sp)} SharePoint CVEs, {pubs[0]} → {pubs[-1]}")
print("  per year: " + ", ".join(f"{y}:{per_year[y]}" for y in sorted(per_year) if y >= 2018))
print(f"  3-year rate {gamma_sp:.3f}/day · last-365 rate {gamma_sp_recent:.3f}/day "
      f"({gamma_sp_recent / gamma_sp:.1f}× — a live surge, not a baseline)")
print(f"  in KEV: {len(sp_kev)}/{len(sp)} = {len(sp_kev) / len(sp):.2%}  "
      f"({(len(sp_kev) / len(sp)) / (len(kev_nvd) / corpus_total):.1f}× the global exploitation rate)")
print(f"  exploitation influx {kev_rate_sp:.4f}/day · disclosure → KEV median {st.median(sp_lags):.0f} d · "
      f"{sum(1 for l in sp_lags if l <= 0)}/{len(sp_lags)} exploited by publication day")
print(f"  network-reachable: {eps_sp:.1%} (n={n_sp})")

SharePoint product names in the CPE dictionary:
  office_sharepoint_server, sharepoint_designer, sharepoint_enterprise_server, sharepoint_foundation, sharepoint_online, sharepoint_portal_server, sharepoint_server, sharepoint_server_client_components_sdk, sharepoint_services, sharepoint_team_services, word_automation_services
  sharepoint_server               588 CVEs   union  588
  sharepoint_foundation           226 CVEs   union  617
  sharepoint_enterprise_server    256 CVEs   union  664
  sharepoint_services              19 CVEs   union  666
  sharepoint_portal_server          6 CVEs   union  668
  sharepoint_team_services          4 CVEs   union  671
  office_sharepoint_server         18 CVEs   union  687
  sharepoint_designer              11 CVEs   union  693



693 SharePoint CVEs, 2003-12-15 → 2026-08-11
  per year: 2018:55, 2019:48, 2020:121, 2021:53, 2022:29, 2023:27, 2024:24, 2025:45, 2026:123
  3-year rate 0.178/day · last-365 rate 0.378/day (2.1× — a live surge, not a baseline)
  in KEV: 20/693 = 2.89%  (6.5× the global exploitation rate)
  exploitation influx 0.0115/day · disclosure → KEV median 158 d · 3/20 exploited by publication day
  network-reachable: 76.9% (n=693)


## 5b. Publication-date rates are undercounts

A window's count is not fixed when the window closes. NVD keeps ingesting records whose publication date falls
inside a window for weeks afterwards, so any rate measured close to the present is low. The size of the effect
is measurable: re-query a window that was already closed when it was first counted.

In [6]:
RECHECK = ("2026-05-07", "2026-08-06", 25_148)      # window, and the count it returned on 2026-08-06
start, end, first_count = RECHECK
later = nvd(f"nvd_window_recheck_{start}_{end[5:]}.json", resultsPerPage=1,
            pubStartDate=f"{start}T00:00:00.000", pubEndDate=f"{end}T23:59:59.999")["totalResults"]
LATE_ARRIVAL = later / first_count - 1
print(f"window {start} .. {end}")
print(f"  counted on {end}: {first_count:,}")
print(f"  counted on {AS_OF}: {later:,}   ({later - first_count:+,} = {LATE_ARRIVAL:+.2%} in "
      f"{(AS_OF - dt.date.fromisoformat(end)).days} days)")
print(f"  -> every gamma derived from publication windows is low by at least this margin, and by more")
print(f"     for the most recent weeks. The direction is always the same: measured rates understate.")

# the corpus grows faster than the windowed rate implies, for the same reason
corpus_growth = (corpus_total - 373_818) / (AS_OF - dt.date(2026, 8, 6)).days
print(f"\ncorpus grew {corpus_total - 373_818:+,} in "
      f"{(AS_OF - dt.date(2026, 8, 6)).days} days = {corpus_growth:.0f}/day, against a windowed "
      f"publication rate of {gamma_global_recent:.0f}/day")
print(f"  the gap is late arrivals plus records published before the last snapshot")

window 2026-05-07 .. 2026-08-06
  counted on 2026-08-06: 25,148
  counted on 2026-08-14: 25,361   (+213 = +0.85% in 8 days)
  -> every gamma derived from publication windows is low by at least this margin, and by more
     for the most recent weeks. The direction is always the same: measured rates understate.

corpus grew +3,403 in 8 days = 425/day, against a windowed publication rate of 288/day
  the gap is late arrivals plus records published before the last snapshot


## 5. MITRE cross-check on disclosure timing

NVD's `published` is when *NVD* published the record, which can trail the CNA. For the SharePoint CVEs that
reached KEV, MITRE's CVE record gives the CNA's own `datePublished` — worth checking before treating
publication-to-KEV lag as a disclosure-to-exploitation interval.

In [7]:
checks = []
for v in sorted(sp_kev, key=lambda v: v["cve"]["cisaExploitAdd"], reverse=True)[:6]:
    cve_id = v["cve"]["id"]
    try:
        rec = fetch(MITRE + cve_id, f"mitre_{cve_id}.json")
    except RuntimeError:
        continue
    meta = rec.get("cveMetadata", {})
    if meta.get("datePublished"):
        gap = (day(v["cve"]["published"]) - day(meta["datePublished"])).days
        checks.append(gap)
        print(f"  {cve_id}  CNA {meta['datePublished'][:10]}  NVD {v['cve']['published'][:10]}  "
              f"Δ {gap:+d} d  KEV {v['cve']['cisaExploitAdd'][:10]}  ({meta.get('assignerShortName', '')})")

if checks:
    print(f"\nNVD lags the CNA by a median of {st.median(checks):.0f} days on this sample — "
          f"small next to the KEV lags above, so NVD dates are used throughout.")

  CVE-2026-50522  CNA 2026-07-14  NVD 2026-07-14  Δ +0 d  KEV 2026-07-22  (microsoft)
  CVE-2026-58644  CNA 2026-07-14  NVD 2026-07-14  Δ +0 d  KEV 2026-07-16  (microsoft)
  CVE-2026-56164  CNA 2026-07-14  NVD 2026-07-14  Δ +0 d  KEV 2026-07-14  (microsoft)
  CVE-2026-45659  CNA 2026-05-22  NVD 2026-05-22  Δ +0 d  KEV 2026-07-01  (microsoft)
  CVE-2026-32201  CNA 2026-04-14  NVD 2026-04-14  Δ +0 d  KEV 2026-04-14  (microsoft)
  CVE-2026-20963  CNA 2026-01-13  NVD 2026-01-13  Δ +0 d  KEV 2026-03-18  (microsoft)

NVD lags the CNA by a median of 0 days on this sample — small next to the KEV lags above, so NVD dates are used throughout.


## 6. Publication history, 2008 to the snapshot

The per-year publication rate drives the transient runs in the model notebook: the equilibria it computes are
targets, and locating the present state requires knowing what the inflow has actually been. NVD caps a
publication-date range at 120 days, so each year is swept in 110-day windows. Cache keys encode the date range
rather than a year index, because the final window of the current year re-splits as the snapshot advances.

Note that the counts are as-measured. Section 5b shows that a closed window keeps growing, so early years are
complete while the most recent are undercounts; the series is not back-corrected.

In [8]:
HISTORY_FROM = 2008
WINDOW_DAYS = 110

history = {}
for year in range(HISTORY_FROM, AS_OF.year + 1):
    start = dt.date(year, 1, 1)
    year_end = min(dt.date(year, 12, 31), AS_OF)
    windows, total = [], 0
    while start <= year_end:
        stop = min(start + dt.timedelta(days=WINDOW_DAYS), year_end)
        n = nvd(f"hist_{start}_{stop}.json", resultsPerPage=1,
                pubStartDate=f"{start}T00:00:00.000", pubEndDate=f"{stop}T23:59:59.999")["totalResults"]
        windows.append([str(start), str(stop), n])
        total += n
        start = stop + dt.timedelta(days=1)
    days = (year_end - dt.date(year, 1, 1)).days + 1
    history[year] = {"total": total, "windows": windows, "days": days}
    print(f"  {year}: {total:>7,} over {days:>3} days = {total / days:>6.1f}/day"
          + ("   (partial year)" if year == AS_OF.year else ""))

Path("data/global_history.json").write_text(json.dumps(history, indent=1))
rate = {y: history[y]["total"] / history[y]["days"] for y in history}
first, last = min(rate), max(rate)
span = last - first
print(f"\nwrote data/global_history.json — {first}–{last}")
print(f"  {rate[first] * 365.25:,.0f}/yr → {rate[last] * 365.25:,.0f}/yr annualized, "
      f"CAGR {(rate[last] / rate[first]) ** (1 / span) - 1:+.1%} over {span} years")
cum = 0
second_diff = []
totals = [history[y]["total"] for y in sorted(history)]
for i in range(2, len(totals)):
    second_diff.append(sum(totals[:i + 1]) - 2 * sum(totals[:i]) + sum(totals[:i - 1]))
print(f"  cumulative {sum(totals):,}; second differences positive in "
      f"{sum(1 for d in second_diff if d > 0)} of {len(second_diff)} years "
      f"(deceleration would be required for the latent pool to be depleting)")

  2008:   5,664 over 366 days =   15.5/day
  2009:   5,778 over 365 days =   15.8/day
  2010:   4,667 over 365 days =   12.8/day
  2011:   4,172 over 365 days =   11.4/day
  2012:   5,351 over 366 days =   14.6/day
  2013:   5,324 over 365 days =   14.6/day
  2014:   8,008 over 365 days =   21.9/day
  2015:   6,595 over 365 days =   18.1/day
  2016:   6,517 over 366 days =   17.8/day
  2017:  18,113 over 365 days =   49.6/day
  2018:  18,154 over 365 days =   49.7/day
  2019:  18,938 over 365 days =   51.9/day
  2020:  19,222 over 366 days =   52.5/day
  2021:  21,950 over 365 days =   60.1/day
  2022:  26,431 over 365 days =   72.4/day
  2023:  30,949 over 365 days =   84.8/day
  2024:  40,704 over 366 days =  111.2/day
  2025:  49,972 over 365 days =  136.9/day
  2026:  51,959 over 226 days =  229.9/day   (partial year)

wrote data/global_history.json — 2008–2026
  5,652/yr → 83,974/yr annualized, CAGR +16.2% over 18 years
  cumulative 348,468; second differences positive in 12 of 17

## 7. From measurements to parameters

Only a few parameters are measured directly. The rest are pinned by *consistency*: given a measured inflow
and a measured stock, the residence time follows. Two such checks come out well, which is the main reason to
trust the resulting sets.

**Retirement $\phi$ against corpus size.** A pool fed at $\gamma$ and drained at $\phi$ settles at
$\gamma/\phi$. A 5-year effective support horizon ($\phi = 1/1825$/day) predicts a standing backlog of
$192 \times 1825 \approx 350{,}000$ CVEs against NVD's actual 373,818 — within 7%. For SharePoint, a 10-year
support horizon predicts $0.155 \times 3650 \approx 566$ against 664 observed. Neither number was fitted.

**Exploitation coefficients from KEV influx.** $\alpha \varepsilon \kappa$ is not observable, but the flux it
produces is: $\alpha_e \varepsilon_e \kappa V_{exist} \approx$ the KEV addition rate. Dividing the measured
influx by the measured backlog gives the product; $\varepsilon$ comes from CVSS reachability and $\kappa$ is
assumed (0.5 globally, 0.9 for SharePoint — a named, actively-targeted product), leaving $\alpha$ as the
residual. The zero-day split uses KEV's own "exploited by publication day" share.

**What stays assumed** (and is flagged as such in the output): $\kappa$; the maturation window $\mu$ (one
patch cycle ≈ 30 days); the removal terms $\delta$, $\rho$, $\tau$, $\lambda$, anchored on CISA's 21-day
median deadline and a ~60-day realistic enterprise remediation time; and the ceiling $K$.

In [9]:
MU = 1 / 30.0                    # a CVE is "new" for about one patch cycle
KAPPA_GLOBAL, KAPPA_SP = 0.50, 0.90
PHI_GLOBAL, PHI_SP = 1 / 1825.0, 1 / 3650.0      # 5-year / 10-year effective support horizon
TAU_DAYS, RHO, DELTA = 14.0, 0.20, 0.003         # patch start, adoption, runtime mitigation
LAM_GLOBAL, LAM_SP = 1 / 2000.0, 1 / 3650.0      # decommissioning
K_GLOBAL, K_SP = 1000.0, 25.0                    # simultaneous-exploitation ceiling


def build(name, gamma, kev_influx, zero_day_frac, eps, kappa, phi, lam, K, backlog_obs, corpus_note):
    """Solve for the exploitation coefficients that reproduce the observed KEV influx."""
    v_new = gamma / MU                              # fresh pool = one cycle of discovery
    v_exist = gamma / phi                           # backlog = inflow x residence time
    influx_exist = kev_influx * (1 - zero_day_frac)
    influx_new = kev_influx * zero_day_frac
    alpha_e = influx_exist / (v_exist * eps * kappa)
    alpha_n = influx_new / (v_new * eps * kappa)
    R = DELTA + RHO / (1 + TAU_DAYS) + lam
    return {
        "name": name, "gamma": gamma, "sigma_S": 0.0, "mu": MU, "phi": phi,
        "alpha_e": alpha_e, "alpha_n": alpha_n, "eps_e": eps, "eps_n": eps, "kappa": kappa,
        "delta": DELTA, "rho": RHO, "tau": TAU_DAYS, "lam": lam, "K": K,
        "_predicted_V_new": v_new, "_predicted_V_exist": v_exist, "_observed_backlog": backlog_obs,
        "_backlog_error": v_exist / backlog_obs - 1.0, "_kev_influx_per_day": kev_influx,
        "_zero_day_share": zero_day_frac, "_R": R, "_mean_days_exploited": 1 / R,
        "_predicted_E": kev_influx / R, "_corpus_note": corpus_note,
    }


PARAM_SETS = {
    "global": build("global software ecosystem", gamma_global, kev_rate_global, zero_day_share,
                    eps_kev, KAPPA_GLOBAL, PHI_GLOBAL, LAM_GLOBAL, K_GLOBAL, corpus_total,
                    f"NVD corpus {corpus_total:,} CVEs on {AS_OF}"),
    "sharepoint": build("Microsoft SharePoint", gamma_sp, kev_rate_sp,
                        sum(1 for l in sp_lags if l <= 0) / len(sp_lags), eps_sp, KAPPA_SP,
                        PHI_SP, LAM_SP, K_SP, len(sp), f"{len(sp)} SharePoint CVEs on {AS_OF}"),
}

for key, ps in PARAM_SETS.items():
    print(f"--- {key}: {ps['name']}")
    print(f"    gamma {ps['gamma']:.3f}/day   mu {ps['mu']:.4f}   phi {ps['phi']:.2e}   "
          f"kappa {ps['kappa']:.2f}   eps {ps['eps_e']:.3f}")
    print(f"    alpha_e {ps['alpha_e']:.3e}/day   alpha_n {ps['alpha_n']:.3e}/day   "
          f"ratio {ps['alpha_n'] / ps['alpha_e']:.0f}x")
    print(f"    R {ps['_R']:.4f}/day → {ps['_mean_days_exploited']:.0f} days in the exploited state")
    print(f"    backlog: predicted {ps['_predicted_V_exist']:,.0f} vs observed "
          f"{ps['_observed_backlog']:,} ({ps['_backlog_error']:+.1%})")
    print(f"    E* ≈ {ps['_predicted_E']:.2f} simultaneously exploited "
          f"({ps['_predicted_E'] / ps['K']:.1%} of K = {ps['K']:,.0f})")

out = {"as_of": str(AS_OF), "sources": {"nvd": NVD, "kev": KEV_FEED, "mitre": MITRE},
       "late_arrival_1wk": LATE_ARRIVAL,
       "measured": {"corpus_total": corpus_total, "cve_per_day_364": gamma_global,
                    "cve_per_day_91": gamma_global_recent, "kev_entries": len(kev_nvd),
                    "kev_share_of_corpus": len(kev_nvd) / corpus_total,
                    "kev_influx_per_day": kev_rate_global, "kev_lag_median_days": st.median(lags),
                    "kev_zero_day_share": zero_day_share,
                    "cisa_deadline_median_days": st.median(due_windows),
                    "network_share_kev": eps_kev, "sharepoint_cves": len(sp),
                    "sharepoint_kev": len(sp_kev), "sharepoint_per_day_1095": gamma_sp,
                    "sharepoint_per_day_365": gamma_sp_recent,
                    "sharepoint_kev_lag_median_days": st.median(sp_lags),
                    "network_share_sharepoint": eps_sp},
       "params": PARAM_SETS}
Path("data/derived_params.json").write_text(json.dumps(out, indent=1))
print("\nwrote data/derived_params.json")

--- global: global software ecosystem
    gamma 197.854/day   mu 0.0333   phi 5.48e-04   kappa 0.50   eps 0.750
    alpha_e 5.170e-06/day   alpha_n 4.664e-05/day   ratio 9x
    R 0.0168/day → 59 days in the exploited state
    backlog: predicted 361,084 vs observed 377,221 (-4.3%)
    E* ≈ 47.73 simultaneously exploited (4.8% of K = 1,000)
--- sharepoint: Microsoft SharePoint
    gamma 0.178/day   mu 0.0333   phi 2.74e-04   kappa 0.90   eps 0.769
    alpha_e 2.165e-05/day   alpha_n 4.649e-04/day   ratio 21x
    R 0.0166/day → 60 days in the exploited state
    backlog: predicted 650 vs observed 693 (-6.2%)
    E* ≈ 0.69 simultaneously exploited (2.8% of K = 25)

wrote data/derived_params.json
